# Exploratory Data Analysis

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 150, "font.size": 10})

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv(Path.cwd().parent / ".temp" / "vehicles.csv")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nBasic statistics:\n{df.describe().round(0)}")

## Missing Values

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).round(1)
missing = missing[missing > 0].sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
missing.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Missing (%)")
ax.set_title("Missing Values by Column")
fig.tight_layout()
plt.show()

## Target Variable: Price

In [ ]:
print("Raw price statistics:")
print(df["price"].describe().round(0))
print(f"\nPrices at $0: {(df['price'] == 0).sum():,}")
print(f"Prices > $100,000: {(df['price'] > 100_000).sum():,}")
print(f"Prices > $1,000,000: {(df['price'] > 1_000_000).sum():,}")

In [ ]:
# Apply domain-informed filters
df = df[(df["price"] >= 500) & (df["price"] <= 80_000)]
df = df[(df["year"] >= 1990) & (df["year"] <= 2021)]
df = df[(df["odometer"] > 0) & (df["odometer"] <= 400_000)]
df = df.dropna(subset=["year", "manufacturer", "fuel", "odometer", "title_status", "transmission"])

df["vehicle_age"] = 2021 - df["year"]
print(f"Filtered shape: {df.shape}")
print(f"\nFiltered price statistics:")
print(df["price"].describe().round(0))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["price"], bins=80, color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Price ($)")
ax.set_ylabel("Count")
ax.set_title("Price Distribution (filtered)")
fig.tight_layout()
plt.show()

## Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df["vehicle_age"], bins=30, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0, 0].set_xlabel("Vehicle Age (years)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_title("Vehicle Age Distribution")

axes[0, 1].hist(df["odometer"], bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0, 1].set_xlabel("Odometer (miles)")
axes[0, 1].set_ylabel("Count")
axes[0, 1].set_title("Odometer Distribution")

top_mfr = df["manufacturer"].value_counts().head(15)
axes[1, 0].barh(top_mfr.index[::-1], top_mfr.values[::-1], color="steelblue")
axes[1, 0].set_xlabel("Count")
axes[1, 0].set_title("Top 15 Manufacturers")

condition_order = ["new", "like new", "excellent", "good", "fair", "salvage"]
df_cond = df[df["condition"].isin(condition_order)]
sns.boxplot(data=df_cond, x="condition", y="price", order=condition_order, ax=axes[1, 1],
            flierprops={"marker": ".", "markersize": 1, "alpha": 0.3})
axes[1, 1].set_xlabel("Condition")
axes[1, 1].set_ylabel("Price ($)")
axes[1, 1].set_title("Price by Condition")
axes[1, 1].tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()

## Correlations

In [ ]:
corr_cols = ["price", "vehicle_age", "odometer"]
corr = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, fmt=".2f")
ax.set_title("Correlation Matrix")
fig.tight_layout()
plt.show()

## Key Observations

- The raw dataset contains 426,880 listings with 26 columns. Significant missingness exists in `condition` (41%), `cylinders` (42%), and `size` (72%). The `county` column is entirely empty.
- Price is heavily right-skewed with extreme outliers (max $3.7B). Filtering to $500–$80,000 retains ~90% of data and removes implausible entries.
- Vehicle age and odometer show clear negative correlations with price (r ≈ −0.58 and r ≈ −0.40 respectively). Age and odometer are positively correlated (r ≈ 0.62).
- Ford, Chevrolet, and Toyota are the three most common manufacturers, together comprising ~37% of listings.
- Vehicle condition shows meaningful price stratification: "like new" and "excellent" listings are priced substantially higher than "fair" or "salvage" listings.